<a href="https://colab.research.google.com/github/MeshalAlsalem/Smart-PC-Matcher/blob/main/project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# SECTION 1: INSTALL DEPENDENCIES & SETUP ENVIRONMENT
# ==============================================================================
!pip -q install \
    "langchain==1.3.14" \
    "langchain-google-genai==4.2.7" \
    "pydantic>=2.13,<3" \
    streamlit

# Download official cloudflared binary
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

import os
import re
import getpass
import time
import subprocess
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Load Gemini API Key
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass.getpass("Enter your Google Gemini API Key: ")

os.environ["GOOGLE_API_KEY"] = api_key.strip()
print("✅ API Key loaded successfully.")

# ==============================================================================
# SECTION 2: GENERATE STREAMLIT APPLICATION FILE (app.py)
# ==============================================================================
app_code = """
import os
import streamlit as st
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Streamlit Page Configuration
st.set_page_config(
    page_title="Smart PC & Laptop Matcher",
    page_icon="🖥️",
    layout="wide"
)

# Initialize Gemini 3.1 Flash Lite Model
MODEL_NAME = "gemini-3.1-flash-lite"
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, thinking_level="low")

# Pydantic Output Schemas
class PCUserNeeds(BaseModel):
    budget_range: str = Field(description="Budget tier selected by user")
    primary_use: str = Field(description="Main usage category")
    portable: bool = Field(description="True if laptop, False for desktop")
    additional_preferences: list[str] = Field(default_factory=list)
    custom_user_notes: str = Field(default="", description="Specific extra requirements")

class PCRecommendation(BaseModel):
    recommended_device: str
    category: str
    short_rationale: str
    match_percentage: int

class FinalTechPlan(BaseModel):
    title: str
    recommended_device: str
    estimated_price_range: str
    match_score: int
    key_specs: list[str]
    why_it_fits: str
    suggested_accessories: list[str]

# Parsers
parser_needs = JsonOutputParser(pydantic_object=PCUserNeeds)
parser_rec = JsonOutputParser(pydantic_object=PCRecommendation)
parser_final = JsonOutputParser(pydantic_object=FinalTechPlan)

TECH_GUIDE = \"\"\"
Use only this guide for PC / Laptop matching based on budget tiers:

- Budget Tier: 1,000 SAR - 2,000 SAR (Refurbished / Entry Student Laptops):
Best for basic programming, light web browsing, document editing, and general academic tasks. (e.g., Lenovo ThinkPad Refurbished / Basic HP Student Laptop).

- Budget Tier: 2,000 SAR - 4,000 SAR (Mid-Range / Entry Gaming Laptops):
Best for Computer Science students, software development, web dev, and light eSports gaming. (e.g., Acer Nitro 5 / HP Victus 15 / Asus Vivobook).

- Budget Tier: 4,000 SAR - 7,000 SAR (High-Performance Laptops & Mid PC Setups):
Best for AAA gaming, heavy compilation, mobile development, multi-monitor setups, and smooth multitasking. (e.g., Lenovo Legion 5 / Apple MacBook Air M2 / Custom Mid Desktop).

- Budget Tier: 7,000 SAR - 12,000 SAR (Pro Creator / High-End Gaming Workstations):
Best for heavy 3D rendering, AI/ML model training, 4K video editing, maxed-out gaming setups, and ultra-portability. (e.g., ASUS ROG Strix / MacBook Pro M3 / RTX 4070 Desktop Setup).

- Budget Tier: 12,000 SAR - 20,000 SAR+ (Ultimate Enthusiast Workstations):
Best for extreme computing, heavy AI research, multi-GPU desktop builds, maximum RAM, and uncompromised performance. (e.g., Custom RTX 4090 Desktop / Max Spec MacBook Pro).
\"\"\"

# LangChain Multi-Step Pipeline with Lightweight Parsers
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the requirement-extraction component of TechAdvisor. Respond strictly in JSON format.\\n{format_instructions}"),
    ("human", "Selected Budget Range: {budget_range}\\nPrimary Use: {use}\\nDevice Type: {dev_type}\\nAdditional Preferences: {extra}\\nCustom User Notes: {custom_notes}")
])
extract_chain = extract_prompt | llm | parser_needs

recommend_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the recommendation component. Match extracted user needs with the provided Tech Guide. Respond strictly in JSON format.\\n{format_instructions}"),
    ("human", "User Needs:\\n{needs_json}\\n\\nTech Guide:\\n{guide}")
])
recommend_chain = recommend_prompt | llm | parser_rec

refine_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Quality Control & Refinement Agent. Optimize and polish the final recommendation for maximum accuracy. Respond strictly in JSON format.\\n{format_instructions}"),
    ("human", "Original User Needs:\\n{needs_json}\\n\\nDraft Recommendation:\\n{rec_json}")
])
refine_chain = refine_prompt | llm | parser_final

def run_pipeline(budget_range, use, dev_type, extra, custom_notes):
    needs = extract_chain.invoke({
        "budget_range": budget_range,
        "use": use,
        "dev_type": dev_type,
        "extra": ", ".join(extra) if extra else "None",
        "custom_notes": custom_notes if custom_notes else "None",
        "format_instructions": parser_needs.get_format_instructions()
    })

    rec = recommend_chain.invoke({
        "needs_json": str(needs),
        "guide": TECH_GUIDE,
        "format_instructions": parser_rec.get_format_instructions()
    })

    final_plan = refine_chain.invoke({
        "needs_json": str(needs),
        "rec_json": str(rec),
        "format_instructions": parser_final.get_format_instructions()
    })

    return final_plan

# Streamlit User Interface
st.title("🖥️ Smart PC & Laptop Matcher")
st.caption("AI-Powered Recommendation System with LangChain & Streamlit")

st.sidebar.header("⚙️ User Preferences")

budget = st.sidebar.selectbox(
    "💰 Select Budget Tier (SAR)",
    [
        "1,000 SAR - 2,000 SAR (Entry Level)",
        "2,000 SAR - 4,000 SAR (Mid-Range / Student)",
        "4,000 SAR - 7,000 SAR (High Performance)",
        "7,000 SAR - 12,000 SAR (Pro / Creator)",
        "12,000 SAR - 20,000 SAR+ (Ultimate Workstation)"
    ],
    index=1
)

use_case = st.sidebar.selectbox(
    "🎯 Primary Use Case",
    [
        "Software Development / Programming",
        "Gaming & Streaming",
        "3D Modeling & Content Creation",
        "Office & General Academic Use"
    ]
)

form_factor = st.sidebar.radio(
    "💻 Form Factor Preference",
    ["Portable Laptop", "Desktop PC Setup"]
)

extras = st.sidebar.multiselect(
    "⚡ Preferred Features",
    [
        "Multi-Monitor Support",
        "High Battery Life",
        "Advanced Cooling",
        "RGB & Aesthetics",
        "32GB+ High Memory (RAM)"
    ],
    default=["Multi-Monitor Support"]
)

custom_notes = st.sidebar.text_area(
    "📝 Custom Notes & Specific Requests",
    placeholder="e.g., Must have Thunderbolt 4, OLED screen preferred, fast delivery, etc."
)

if st.sidebar.button("🔍 Find Match", type="primary"):
    with st.spinner("Analyzing requirements & refining match..."):
        result = run_pipeline(budget, use_case, form_factor, extras, custom_notes)

        st.success(f"### 🎯 {result.get('title', 'Recommendation Result')}")

        st.markdown(f"#### 💻 **Recommended Device:** {result.get('recommended_device', 'N/A')}")
        st.write("")

        col1, col2 = st.columns(2)
        with col1:
            st.metric("Estimated Price Range", result.get('estimated_price_range', 'N/A'))
        with col2:
            st.metric("Match Score", f"{result.get('match_score', 90)}%")

        st.subheader("💡 Why it fits:")
        st.write(result.get('why_it_fits', ''))

        st.subheader("🛠️ Key Specs:")
        for spec in result.get('key_specs', []):
            st.markdown(f"- {spec}")

        st.subheader("🎧 Suggested Accessories:")
        for acc in result.get('suggested_accessories', []):
            st.markdown(f"- {acc}")
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("📝 app.py generated successfully.")

# ==============================================================================
# SECTION 3: LAUNCH STREAMLIT SERVER & PRINT CLEAN PUBLIC URL
# ==============================================================================
# Kill old background processes
!pkill -f streamlit
!pkill -f cloudflared

# Launch Streamlit server in background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(4)

# Launch Cloudflare Tunnel
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8501"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("⏳ Establishing stable Cloudflare Tunnel...")

# Parse and display clean URL after full registration
clean_url = None
for _ in range(40):
    line = tunnel.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            clean_url = match.group(0)
            break

if clean_url:
    print("⏳ Waiting 3 seconds for DNS propagation...")
    time.sleep(3)
    print("\n" + "="*60)
    print("🚀 STREAMLIT IS LIVE! Click the link below:")
    print(clean_url)
    print("="*60 + "\n")
else:
    print("⚠️ Could not retrieve URL. Please rerun this cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 25.9 MB/s eta 0:00:00
✅ API Key loaded successfully.
📝 app.py generated successfully.
⏳ Establishing stable Cloudflare Tunnel...
⏳ Waiting 3 seconds for DNS propagation...

🚀 STREAMLIT IS LIVE! Click the link below:
https://zealand-seal-namespace-die.trycloudflare.com

